# Notebook 1: The Listings Endpoint

## What this notebook covers

The listings endpoint is one of the main data sources on the Domain API. It returns individual property records, one per property, with fields like address, price, listing date, property type, and number of bedrooms.

This notebook teaches you:

1. How the listings endpoint works and how to structure a data access request.
2. How to apply filters (listing type, property type, date range).
3. How to detect the total record count with a density probe.
4. How to retrieve all records page by page for areas with up to 1,000 listings.
5. How to save your results to a CSV file.

For requests with more than 1,000 listings, a proper tutorial is provided in Notebook 3.

**By the end** you will have a complete dataset for two Melbourne suburbs stored in a DataFrame and saved as a CSV file.

---

## Before You Start

**If you have already completed Notebook 0**, your packages, `.env` file, and connection are already set up. Skip to the *Key Terms* section below.

If this is your first exposure to Domain API, complete `notebook-0-getting-started.ipynb` before this notebook. It covers package installation, creating the `.env` credentials file, and making a test call to confirm your connection works.

---
## Key Terms

**API (Application Programming Interface):** A way for your computer to ask another computer for data over the internet. Think of it as placing a phone order at a restaurant rather than walking in.

**Endpoint:** A specific address within an API that handles one type of request. This notebook uses the residential listings search endpoint.

**Credit:** One unit of API usage. Every request costs 1 credit, regardless of how many records it returns. Your AURIN account has 1,000 credits per month.

**Record:** One row of data. In this notebook, one property listing.

**POST request:** The request method used by the listings endpoint. You send a JSON body describing your search criteria, and the server responds with matching records.

**Pagination:** When a large set of results is split across multiple pages. You request page 1, then page 2, and so on.

**JSON:** A text format for storing structured data. The API sends responses and accepts requests in JSON format.

**DataFrame:** A table of data in Python, similar to an Excel spreadsheet. Rows are listings, columns are fields (suburb, price, date, etc.).

---

## The Listings Endpoint: What It Returns

The endpoint address is:

```
POST /v1/listings/residential/_search
```

You send it a JSON body describing your search criteria. It responds with a list of property records.

**Key fields in each record:**

| Field path | Description |
|---|---|
| `id` | Unique listing identifier |
| `listingType` | `Sold` or `Rent` |
| `dateListed` | Date the listing appeared on Domain |
| `soldData.soldPrice` | Sale price (sold listings only) |
| `propertyDetails.suburb` | Suburb name |
| `propertyDetails.propertyType` | Property type (see note below) |
| `propertyDetails.bedrooms` | Number of bedrooms |
| `propertyDetails.latitude` | Latitude coordinate (used in Notebook 4 for spatial queries) |
| `propertyDetails.longitude` | Longitude coordinate |


> **Property types on this endpoint:** the listings endpoint returns a wide range of values here, including `House`, `ApartmentUnitFlat`, `Townhouse`, `Villa`, `Land`, `Rural`, and others. This is different from the suburb statistics endpoint in Notebook 2, which only accepts `House` or `Unit` as a filter. For the full list of values, see the *Agents & Listings API - Data Dictionary* in the metadata.

The API wraps records in group objects in its response. The `fetch_page` function later in this notebook unpacks that structure automatically.

---
## The 1,000-Record Cap: Why It Matters

Imagine you ask a librarian for every book published between 2019 and 2024 on a particular
topic. The librarian hands you a pile of 1,000 books and says nothing. You take them home
assuming that is all there is. But there were actually 4,200 books, the librarian quietly
held back 3,200 without any warning.

That is exactly what the Domain API does. If your query matches 4,200 records, you get
the first 1,000. The remaining 3,200 are silently dropped. No error. No warning. No note
saying "we held some back."

The fix is to always ask the librarian "how many books match?" before deciding how to
retrieve them. That is the density probe.

---
## Setup: Check Your Packages

Run the cell below to confirm all required packages are installed. You should see
"All packages found. Setup OK." at the end. If any package is missing, the cell
will tell you which one and what command to run to install it.

In [ ]:
# Check that all required packages are installed before going further
missing = []

try:
    import requests
except ImportError:
    missing.append('requests')

try:
    import pandas
except ImportError:
    missing.append('pandas')

try:
    import dotenv
except ImportError:
    missing.append('python-dotenv')

if missing:
    print('The following packages are missing:')
    for pkg in missing:
        print(f'  {pkg}')
    print()
    print('Open a terminal and run:')
    print(f'  pip install {" ".join(missing)}')
    print('Then restart the Jupyter kernel (Kernel menu > Restart) and run this cell again.')
else:
    print('All packages found. Setup OK.')

---
## Setup: Connect to the API

The cell below loads your credentials from the `.env` file and connects to the AURIN proxy.
It prints your username so you can confirm the right account is being used. Your password
is never printed.

**What `sys.path.insert` does:** it tells Python where to find the `utils.py` helper file.
Because both files are in the same folder, `'.'` (meaning "the current directory") is all
that is needed.

**What `from utils import` does:** it loads the authentication setup and helper functions
that are shared across all four notebooks, so you do not need to repeat that code here.

In [ ]:
import sys
import pandas as pd

# Tell Python to look in the current directory for utils.py
if '.' not in sys.path:
    sys.path.insert(0, '.')

# Import the shared helper functions and authentication setup
from utils import PROXY_BASE, APICallTracker, probe_count

# Create a tracker that will count every API call we make
tracker = APICallTracker()

# Print the proxy address and username to confirm the connection is configured
# (AURIN_USERNAME is loaded inside utils.py from your .env file)
import os
print(f'Connected to: {PROXY_BASE}')
print(f'Account:      {os.getenv("AURIN_USERNAME", "(not found -- check your .env file)")}')

**Expected output:** Two lines. The first shows the proxy URL. The second shows your
AURIN email address. If you see "not found -- check your .env file", re-read Step 3
in the Before You Start section above and confirm the `.env` file is in the main
`domain_api` folder (not inside `training-materials/phase-2/`).

---
## Step 1: The Density Probe

Before downloading anything, run a quick 1-credit check to find out how many records
your query would return. This is like asking "how many books match my search?" before
deciding how to retrieve them.

The probe sends a request with `pageSize=1` (asking for just one record). The API
processes the full query and tells you the total match count in a response header
called `X-Total-Count`. The probe function reads that header and returns the number.

We will probe two suburbs:
- **South Yarra 3141** (inner Melbourne, broad date window) -- expected to exceed 1,000
- **Williamstown 3016** (outer Melbourne, narrower window) -- expected to be under 1,000

### Define the queries

First, define the search parameters for each suburb. Read the comments carefully --
each line explains what it does. You will not spend any credits yet.

In [ ]:
# Define the search for South Yarra (dense suburb, broad date window)
dense_payload = {
    'listingType': 'Sold',       # we want properties that have already sold
    'listedSince': '2019-01-01', # only include listings that appeared from this date onwards
    'locations': [{
        'state': 'VIC',
        'suburb': 'South Yarra',  # suburb name -- must match Domain exactly
        'postCode': '3141',
        'includeSurroundingSuburbs': False,  # do not include neighbouring suburbs
    }],
}

# Define the search for Williamstown (sparser suburb, narrower date window)
sparse_payload = {
    'listingType': 'Sold',
    'listedSince': '2023-01-01',  # shorter window keeps count under 1,000
    'locations': [{
        'state': 'VIC',
        'suburb': 'Williamstown',
        'postCode': '3016',
        'includeSurroundingSuburbs': False,
    }],
}

print('Queries defined. No API calls made yet.')

### Run the probe

The next cell runs both probes. **This costs 2 credits total** (1 per probe).
The output will show the record count for each suburb.

Because this is a live API, the counts you see might be higher than the example figures shown in this notebook, new listings are added continuously. What matters is whether your count is above or below 1,000, not the exact number.

In [ ]:
# Run the density probe for the dense suburb (1 credit)
dense_count = probe_count(dense_payload, tracker)
tracker.checkpoint('Probes')

# Run the density probe for the sparse suburb (1 credit)
sparse_count = probe_count(sparse_payload, tracker)
tracker.checkpoint('Probes')

print(f'South Yarra 3141 (from 2019): {dense_count:,} records')
print(f'Williamstown 3016 (from 2023): {sparse_count:,} records')

**What the numbers mean:**

- If a count is **1,000 or less**, simple pagination (Step 2) will retrieve all records in a straightforward page-by-page loop.
- If a count is **over 1,000**, simple pagination will silently miss records above position 1,000. Notebook 3 (Pagination Tricks) covers the cursor advancement technique needed for those cases.

The probe is never wasted. Even if the count turns out to be small, 1 credit to confirm completeness is far cheaper than building a study on unknowingly incomplete data.

---
## Step 2: Simple Pagination for Small Areas

When the probe count is 1,000 or fewer, a page-by-page loop retrieves all records.
Think of it like flipping through a short catalogue: you read page 1, page 2, and so
on until you reach the last page. Each page holds up to 200 records.

The loop stops when a page comes back with fewer records than the page size, or with
no records at all. The second condition handles the case where the total is an exact
multiple of 200. Without it, the loop would make one extra unnecessary API call.

### The `fetch_page` function

The cell below defines a helper function that fetches one page at a time. Read the
comments to understand each line. You will not call this function yet.

In [ ]:
def fetch_page(base_payload, page, page_size=200):
    """Fetch a single page of listings from the search endpoint.

    base_payload: the search criteria (suburb, date range, listing type).
    page:         which page number to retrieve (1 = first page).
    page_size:    how many records per page (maximum 200).

    Returns: (list of listing dicts, True if more pages exist, False if this was the last).
    """
    # Add page number and page size to the query
    payload = {**base_payload, 'pageSize': page_size, 'pageNumber': page}

    # Send the request to the API
    r = tracker.post(
        f'{PROXY_BASE}/v1/listings/residential/_search',
        json_body=payload,
    )

    # If the request failed, print the error and return nothing
    if r.status_code != 200:
        print(f'  Page {page} failed: HTTP {r.status_code}')
        return [], False

    # Extract the listings from the response JSON
    groups = r.json()  # the API wraps listings in groups (e.g. projects)
    listings = []
    for g in groups:
        raw = g.get('listings') or g.get('listing')
        if isinstance(raw, list):
            listings.extend(raw)   # add a list of listings
        elif isinstance(raw, dict):
            listings.append(raw)   # add a single listing

    # Check if there are more pages by comparing how many groups came back
    # to the page size -- if fewer came back, this was the last page
    actual_page_size = int(r.headers.get('x-pagination-pagesize', page_size))
    has_more = len(groups) >= actual_page_size

    return listings, has_more


print('fetch_page defined.')

### The `fetch_all_simple` function

This function calls `fetch_page` in a loop until there are no more pages. It collects
all the results into one list. Run the cell to define it.

In [ ]:
def fetch_all_simple(base_payload):
    """Retrieve all listings for a query that returns 1,000 or fewer total records.

    Loops through pages until no more are available. Prints progress as it goes.
    Do NOT use this for queries that exceed 1,000 records -- use fetch_all_cursor instead.

    base_payload: the search criteria dict.
    Returns: a list of all listing dicts.
    """
    all_listings = []   # this list will grow as we collect each page
    page = 1

    while True:  # keep looping until we explicitly break out
        batch, has_more = fetch_page(base_payload, page)
        all_listings.extend(batch)  # add this page's results to the running total
        print(f'  Page {page}: +{len(batch)} records | total so far: {len(all_listings)}')

        # Stop if the page was empty, or if the API signals no more pages exist.
        # The empty-batch check handles the edge case where the total record count
        # is an exact multiple of page_size (e.g. exactly 200, 400, 600 ...): in
        # that case the final real page looks "full" and has_more is True, so without
        # this check the loop would make one extra credit-costing call that returns
        # nothing. Checking for an empty batch catches that and exits cleanly.
        if not batch or not has_more:
            break

        page += 1  # move to the next page

    return all_listings


print('fetch_all_simple defined.')

### Run simple pagination on Williamstown

The cell below fetches all records for Williamstown 3016 using simple pagination.
It then builds a DataFrame (a table) from the results so you can inspect them.

**Expected output:** You should see one or more lines like "Page 1: +200 records",
followed by a table showing the first five listings.

In [ ]:
print('Fetching Williamstown 3016 (simple pagination)...')
williamstown_raw = fetch_all_simple(sparse_payload)
tracker.checkpoint('Williamstown simple fetch')

# Build a DataFrame from the raw listing data
# Each listing is a deeply nested dictionary; we pull out the key fields here
df_willi = pd.DataFrame([
    {
        'id':            item.get('id'),
        'suburb':        (item.get('propertyDetails') or {}).get('suburb'),
        'property_type': (item.get('propertyDetails') or {}).get('propertyType'),
        'bedrooms':      (item.get('propertyDetails') or {}).get('bedrooms'),
        'date_listed':   (item.get('dateListed') or '')[:10],  # keep only the date part
        'sold_price':    (
            (item.get('soldData') or {}).get('soldPrice')
            or (item.get('priceDetails') or {}).get('price')
        ),
    }
    for item in williamstown_raw
])

print(f'\nWilliamstown: {len(df_willi):,} records fetched.')
df_willi.head()  # show the first 5 rows

### What else is in each record?

The DataFrame above only extracts six fields, but each listing contains much more. A few fields worth knowing about:

| Field path | Description |
|---|---|
| `soldData.soldDate` | The date the property actually sold (different from `dateListed`, which is when the listing appeared on Domain) |
| `soldData.saleMethod` | How it sold: `SoldByAuction`, `SoldByPrivateTreaty`, `SoldPriorToAuction`, `Withdrawn`, or `NotStated` |
| `propertyDetails.displayableAddress` | Full street address as a single string |
| `propertyDetails.bathrooms` | Number of bathrooms |
| `propertyDetails.carspaces` | Number of car spaces |
| `propertyDetails.landArea` | Land area in square metres |
| `propertyDetails.buildingArea` | Building area in square metres |
| `propertyDetails.features` | List of features, e.g. `['AirConditioning', 'Ensuite']` |
| `headline` | The listing title as it appeared on Domain |

**Why `[:10]` is enough for the date:** date fields like `dateListed` and `soldData.soldDate` come back as ISO 8601 datetime strings (For example `2023-04-07T00:00:00`). The first 10 characters are always the date (`2023-04-07`); the time portion is always midnight and carries no information for sold listings. Taking `[:10]` strips the time and gives a clean date string.

Run the cell below to inspect a single raw record and see the full structure returned by the API.

In [ ]:
import json

# Print the first raw listing record so you can see everything the API returns.
# No API call is made here -- this is just inspecting data already fetched above.
print(json.dumps(williamstown_raw[0], indent=2))

---

## Step 3: Filter Your Search

The payload accepts several filter fields. The most commonly used ones are:

| Field | Example value | Effect |
|---|---|---|
| `listingType` | `'Sold'`, `'Sale'`, `'Rent'`, `'Share'`, `'NewHomes'` | Limit to a specific listing category (see note below) |
| `propertyTypes` | array of property type values | Limit to specific property types (see note below) |
| `minBedrooms` | `2` | Only listings with at least this many bedrooms |
| `maxBedrooms` | `4` | Only listings with at most this many bedrooms |
| `listedSince` | `'2022-01-01'` | Only listings that appeared from this date onwards |

> **`listingType` values:** `Sold` returns completed sales (what most research uses). `Sale` returns properties currently listed for sale but not yet sold. `Rent` returns rental listings. `Share` returns share accommodation. `NewHomes` returns new home listings. For the full list of filter fields and allowed values, see the *Agents & Listings API - Data Dictionary* in the `metadata/data dictionary (from domain)/` folder.

> **`propertyTypes` filter values:** the API returns property types in PascalCase (e.g. `ApartmentUnitFlat`, `Townhouse`, `House`). See the full list in the response field table in the previous section. Whether the filter accepts these same values, or different ones, is not documented in the schema. Check the *Agents & Listings API - Data Dictionary* for confirmed filter values before using this field, and verify your results against your probe count.

The cell below adds a bedroom filter to the Williamstown query and shows how the record count changes. Filtering costs the same as an unfiltered probe (1 credit per check).

In [ ]:
# Add a bedroom filter to the Williamstown search
filtered_payload = {
    **sparse_payload,   # start with the same criteria as before
    'minBedrooms': 2,   # only properties with at least 2 bedrooms
    'maxBedrooms': 4,   # only properties with at most 4 bedrooms
}

# Check how many listings match the filtered criteria (1 credit)
filtered_count = probe_count(filtered_payload, tracker)
tracker.checkpoint('Filtered probe')

print(f'Williamstown (all bedrooms):    {sparse_count:,} records')
print(f'Williamstown (2 to 4 bedrooms): {filtered_count:,} records')
print()
print(f'Excluded: {sparse_count - filtered_count:,} records (studios, 1-bed, and 5-bed-plus properties)')

---

## Checking Your Work

Compare the number of records you fetched against the probe count from Step 1. They should be close. A small difference (a few records) is normal because new listings can appear between the probe and the full fetch.

A large difference (more than 10%) suggests the date window in the payload changed between the probe and the fetch, or that filters were applied inconsistently.

In [ ]:
# Compare the probe count to the actual fetch count
print(f'Probe count (from Step 1):    {sparse_count:>6,}')
print(f'Fetched count:                {len(df_willi):>6,}')
print(f'Difference:                   {abs(sparse_count - len(df_willi)):>6,}  (a small number is expected)')
print()

# Check for duplicate records (there should be none)
duplicates = df_willi['id'].duplicated().sum()
print(f'Duplicate listing IDs: {duplicates}  (expected: 0)')
print()

print(f'Date range covered:')
print(f'  Earliest listing: {df_willi["date_listed"].min()}')
print(f'  Latest listing:   {df_willi["date_listed"].max()}')

---

## Saving Your Results

Once you have a DataFrame, save it as a CSV file. A CSV is a plain-text spreadsheet that Excel and other tools can open. Run the cell below to save the Williamstown dataset.

> **Save early, save often.** Data fetched from the API only exists in memory while the notebook is running. If the notebook crashes, the kernel restarts, or your computer shuts down, that data is gone and you will need to call the API again, spending credits you have already used. Get into the habit of saving to a CSV file immediately after every fetch, before doing any further analysis.

In [ ]:
# Save the dataset to a CSV file
# index=False means do not write the row numbers as a column
# If you save somewhere other than this folder, give the full path, using
# forward slashes (/) or doubled backslashes (\\). A pasted Windows path with
# single backslashes may not work.
output_filename = 'williamstown_sold_listings.csv'
df_willi.to_csv(output_filename, index=False)

print(f'Saved: {output_filename}')
print(f'Rows:    {len(df_willi):,}')
print(f'Columns: {list(df_willi.columns)}')
print()
print('You can open this file in Excel or import it into R, QGIS, or STATA.')

---

## What If Something Goes Wrong?

**"not found -- check your .env file" when loading credentials:**
The `.env` file is missing or in the wrong location. It must be in the main `domain_api` folder, not inside `training-materials/phase-2/`. Check that the file name starts with a dot and has no other extension (`.env.txt` will not work).

**Probe returns 0 or the fetch returns an empty DataFrame:**
The suburb name or postcode may be incorrect. Domain's website shows the correct spelling. Verify that `suburb` and `postCode` in the payload match exactly, including capitalisation. For example, "south yarra" will not work; it must be "South Yarra".

**The fetch count is much lower than the probe count:**
The probe and full fetch were run far apart in time, and new listings appeared between them. A difference of more than 10% may also indicate that the date window in the payload changed between the probe and the fetch. Re-run the probe to confirm.

**The probe returns more than 1,000 records:**
Simple pagination will silently miss records above position 1,000. See Notebook 3 (Pagination Tricks) for the cursor advancement technique that handles this case.

---

## What's Next: Notebook 2

You can now query the listings endpoint, apply filters, retrieve all pages for areas with up to 1,000 records, and save the results to a CSV file.

**Notebook 2** covers a completely different endpoint: aggregated market statistics. Instead of individual property records, it returns one row per quarter, summarising the median sale price, the number of properties sold, and average days on market across an entire suburb. One call retrieves up to 11 years of quarterly history for the same cost as a single listings probe.

→ Open `notebook-2-suburb-statistics.ipynb` to continue.

---
## Credit Summary

The table below shows how many API credits were consumed in each section of this
notebook. Each row corresponds to a checkpoint label you saw printed during the run.

Credits reset to 1,000 on the first of each calendar month. You can check your
remaining balance on the AURIN dashboard. There is no programmatic way to check
the balance -- dashboard checks are a required part of the workflow before any
large extraction.

In [ ]:
tracker.summary()